In [0]:
windowfn_df = spark.read.format("csv").option("header",True).option("inferSchema",True).load('abfss://unitycatalog@databricksstudysa.dfs.core.windows.net/catalog/windowdatamodified.csv')

In [0]:
windowfn_df.show()

In [0]:
windowfn_df.orderBy("country","invoicevalue").show(50)

Let's try to find the running total

3 things are required when calculating running total

    - Partition Column
    - Sorting Column
    - Window Size

In [0]:
from pyspark.sql import *

In [0]:
my_window_size = Window.partitionBy("country").orderBy("weeknum").rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [0]:
from pyspark.sql.functions import sum

In [0]:
from pyspark.sql import functions as F

result_df = windowfn_df.withColumn("running_total", F.sum("invoicevalue").over(my_window_size))


In [0]:
result_df.show()

In [0]:
from pyspark.sql.functions import desc
my_window = Window.partitionBy("country").orderBy(desc("invoicevalue"))


In [0]:
from pyspark.sql.functions import rank, dense_rank,row_number

In [0]:
final_df = windowfn_df.withColumn("rank",rank().over(my_window))

In [0]:
final_df.show()

In [0]:
final_dense_df = windowfn_df.withColumn("dense_rank",dense_rank().over(my_window))

In [0]:
final_dense_df.show()

In [0]:
final_dense_df = windowfn_df.withColumn("row_num",row_number().over(my_window))

In [0]:
final_dense_df.show()

In [0]:
final_dense_df.select("*").filter(F.col("row_num") == 1).show()